# TB Pipeline — SOTA Comparison

Builds the consolidated SOTA comparison table for the paper. Combines:

| # | Method | Source | Notes |
|---|---|---|---|
| 1 | Liu 2020 TBX11K classifier | Reported (paper) | TB AUROC 0.958 on TBX11K |
| 2 | Liu 2020 TBX11K bbox detector | Reported (paper) | mAP@0.5 = 0.508 — referenced as 'no pixel-mask SOTA' |
| 3 | MedSAM zero-shot lung segmentation | Reported (paper) | Dice ~0.85 on CXR lung |
| 4 | Raw TXV TB-mimic max | Run here | Max of (Consolidation, Infiltration) probs — TXV-only baseline, no our head |
| 5 | Our Baseline path | From eval_baseline.ipynb | Grad-CAM lesion |
| 6 | Our MoE path | From eval_moe.ipynb | Pathology-specialised experts |

**Note on vanilla MedSAM lesion:** intentionally excluded. MedSAM is a *promptable* segmentation model trained on lung/organ masks — never on TB lesions. Without a credible prompting protocol it cannot produce lesion masks, so any number we report would be misleading. We use MedSAM only as the lung-segmentation reference.

## Datasets to attach
Same as `eval_moe.ipynb`.

In [8]:
# Cell 1: Clone repo + install
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/repo')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '-b', 'cleaned-repo',
         'https://github.com/mabdullahi7780/dl-project-codebase.git', str(REPO_DIR)],
        check=True
    )
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Repo ready:', REPO_DIR)

Repo ready: /kaggle/working/repo


In [9]:
# Cell 2: Paths + device
from pathlib import Path
import torch

KAGGLE_INPUT = Path('/kaggle/input')
MONTGOMERY_ROOT  = KAGGLE_INPUT / 'datasets/iahmedhabib/montgomery/montgomery'
SHENZHEN_ROOT    = KAGGLE_INPUT / 'datasets/iahmedhabib/shehzhenn/shenzhen'
TBX11K_ROOT      = KAGGLE_INPUT / 'datasets/usmanshams/tbx-11/TBX11K'
NIH_ROOT         = KAGGLE_INPUT / 'datasets/organizations/nih-chest-xrays/data'
CHECKPOINTS_ROOT = KAGGLE_INPUT / 'datasets/iahmedhabib/tb-pipeline-checkpoints'
MEDSAM_CKPT      = KAGGLE_INPUT / 'datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = Path('/kaggle/working/sota')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', device)
print('Output dir:', OUT_DIR)

Device: cuda
Output dir: /kaggle/working/sota


## Run #4 — Raw TXV TB-mimic max

Run TorchXRayVision's pretrained DenseNet121 on the same held-out 560-image split. For each image, take `max(P[Consolidation], P[Infiltration])` as the TB-likelihood proxy and compute per-dataset AUROC.

This isolates 'what does TXV give you out of the box, without any TB-specific fine-tuning?'.

In [10]:
# Build the same eval split moe_eval uses, then run raw TXV through it.
import yaml, numpy as np, pandas as pd, torch
from pathlib import Path
from src.evaluation.baseline_eval import (
    DOMAIN_IDS, build_eval_manifest, make_test_splits,
    _load_image_for_entry, _safe_auroc,
)
from src.components.component0_qc import harmonise_sample
from src.components.component2_txv import Component2SoftDomainContext, TXV_CLASS_NAMES

# Build the same manifest + split as eval_moe
samples = build_eval_manifest(
    montgomery_root=MONTGOMERY_ROOT, shenzhen_root=SHENZHEN_ROOT,
    tbx11k_root=TBX11K_ROOT, nih_root=NIH_ROOT if NIH_ROOT.exists() else None,
    tbx_list_name='all_trainval.txt', nih_cache_path=OUT_DIR/'_nih_index_cache.json',
)
splits = make_test_splits(
    samples, seed=42, holdout_frac=0.2, limit_per_domain=200,
    cache_path=OUT_DIR/'test_splits.json',
)
for d in DOMAIN_IDS:
    print(f'  {d:<11}: {len(splits.get(d, []))} held-out')

# Load TXV with NO custom routing head, NO TB head — purely the pretrained pathology probs
c2 = Component2SoftDomainContext(backend='auto').to(device)
c2.eval()
print('TXV active backend:', c2.active_backend)
print('TXV class names:', TXV_CLASS_NAMES)

# Class indices for TB-mimic (presence of one of these → TB-suspicious).
# TXV_CLASS_NAMES is lowercase ('consolidation','infiltration'), so match case-insensitively.
MIMIC_CLASSES = ['consolidation', 'infiltration']
_lower_names = [c.lower() for c in TXV_CLASS_NAMES]
mimic_idx = [_lower_names.index(c) for c in MIMIC_CLASSES if c in _lower_names]
assert len(mimic_idx) > 0, (
    f'No TB-mimic classes found in TXV_CLASS_NAMES={TXV_CLASS_NAMES}. '
    f'Looked for {MIMIC_CLASSES}.'
)
print('TB-mimic indices:', {TXV_CLASS_NAMES[i]: i for i in mimic_idx})

rows = []
for dom in DOMAIN_IDS:
    for entry in splits.get(dom, []):
        if entry.tb_label is None:  # NIH has no TB labels
            continue
        try:
            image_np, _ = _load_image_for_entry(entry)
            harm = harmonise_sample({'image': image_np, 'dataset_id': entry.dataset_id})
            x224 = harm.x_224.unsqueeze(0).to(device)
            with torch.no_grad():
                txv = c2.forward_features(x224)
            probs = torch.sigmoid(txv.pathology_logits).squeeze(0).cpu().numpy()
            tb_score = float(probs[mimic_idx].max())
            rows.append({'dataset': dom, 'tb_label': int(entry.tb_label), 'tb_score': tb_score})
        except Exception as e:
            print(f'  skip {dom}/{entry.image_path}: {e}')

df_raw = pd.DataFrame(rows)
df_raw.to_csv(OUT_DIR/'raw_txv_per_image.csv', index=False)
print(f'\nWrote {len(df_raw)} per-image scores')

raw_auroc = {}
for dom in DOMAIN_IDS:
    sub = df_raw[df_raw['dataset']==dom]
    if len(sub) < 2 or len(set(sub['tb_label'])) < 2:
        raw_auroc[dom] = None
        continue
    raw_auroc[dom] = _safe_auroc(sub['tb_label'].to_numpy(), sub['tb_score'].to_numpy())

print('\nRaw TXV TB-mimic max — AUROC by dataset:')
for d, v in raw_auroc.items():
    print(f'  {d:<11}: {v if v is None else f"{v:.4f}"}')

  montgomery : 28 held-out
  shenzhen   : 132 held-out
  tbx11k     : 200 held-out
  nih_cxr14  : 200 held-out
TXV active backend: xrv
TXV class names: ('atelectasis', 'consolidation', 'infiltration', 'pneumothorax', 'edema', 'emphysema', 'fibrosis', 'effusion', 'pneumonia', 'pleural_thickening', 'cardiomegaly', 'nodule', 'mass', 'hernia', 'lung_lesion', 'fracture', 'lung_opacity', 'enlarged_cardiomediastinum')
TB-mimic indices: {'consolidation': 1, 'infiltration': 2}

Wrote 339 per-image scores

Raw TXV TB-mimic max — AUROC by dataset:
  montgomery : 0.5816
  shenzhen   : 0.6901
  tbx11k     : 0.3459
  nih_cxr14  : None


## Build the consolidated SOTA table

Combines reported numbers + raw TXV + our eval CSVs. Outputs `sota_comparison.csv` ready for paper import.

In [11]:
import pandas as pd
from pathlib import Path

BASELINE_DIR = Path('/kaggle/working/eval_baseline')
MOE_DIR      = Path('/kaggle/working/eval_moe')

def _val(df, metric, dataset):
    sub = df[(df['metric']==metric) & (df['dataset']==dataset)]
    if sub.empty: return None
    try: return float(sub['value'].iloc[0])
    except (ValueError, TypeError): return None

rows = []

# 1) Liu 2020 TBX11K classifier
rows.append({'method': 'Liu et al. 2020 (TBX11K classifier)', 'source': 'paper',
             'metric': 'TB AUROC', 'dataset': 'TBX11K', 'value': 0.958, 'notes': 'reported best detection AUC'})

# 2) Liu 2020 detector — mAP@0.5
rows.append({'method': 'Liu et al. 2020 (TBX11K bbox detector)', 'source': 'paper',
             'metric': 'mAP@0.5', 'dataset': 'TBX11K', 'value': 0.508, 'notes': 'TB-X box detection (different metric — referenced for context)'})

# 3) MedSAM zero-shot lung
for dom in ['Shenzhen', 'Montgomery']:
    rows.append({'method': 'MedSAM zero-shot (Ma 2024)', 'source': 'paper',
                 'metric': 'Lung Dice', 'dataset': dom, 'value': 0.85, 'notes': 'reported approx — CXR lung segmentation'})

# 4) Raw TXV TB-mimic max — from this notebook
for dom_key, dom_pretty in [('montgomery','Montgomery'), ('shenzhen','Shenzhen'), ('tbx11k','TBX11K')]:
    rows.append({'method': 'Raw TXV (Cohen 2022) TB-mimic max', 'source': 'this notebook',
                 'metric': 'TB AUROC', 'dataset': dom_pretty,
                 'value': raw_auroc.get(dom_key), 'notes': 'no fine-tuning; max(Consolidation,Infiltration)'})

# 5) Our Baseline path — from eval_baseline
if BASELINE_DIR.exists():
    bl_sys = pd.read_csv(BASELINE_DIR/'baseline_system.csv')
    bl_cmp = pd.read_csv(BASELINE_DIR/'baseline_components.csv')
    for dom_key, dom_pretty in [('montgomery','Montgomery'), ('shenzhen','Shenzhen'), ('tbx11k','TBX11K')]:
        rows.append({'method': 'Ours — Baseline (Grad-CAM)', 'source': 'eval_baseline',
                     'metric': 'TB AUROC', 'dataset': dom_pretty,
                     'value': _val(bl_sys, 'tb_head_auroc', dom_key), 'notes': 'trained TB head'})
        rows.append({'method': 'Ours — Baseline (Grad-CAM)', 'source': 'eval_baseline',
                     'metric': 'Timika AUROC', 'dataset': dom_pretty,
                     'value': _val(bl_sys, 'timika_auroc', dom_key), 'notes': 'lesion path: Grad-CAM'})
    for dom_key, dom_pretty in [('montgomery','Montgomery'), ('shenzhen','Shenzhen')]:
        rows.append({'method': 'Ours — Baseline (Grad-CAM)', 'source': 'eval_baseline',
                     'metric': 'Lung Dice', 'dataset': dom_pretty,
                     'value': _val(bl_cmp, 'c4_lung_dice', dom_key), 'notes': 'fine-tuned C4 decoder'})
else:
    print('WARNING: eval_baseline outputs not found — skipping baseline rows. Run eval_baseline.ipynb first.')

# 6) Our MoE path — from eval_moe
if MOE_DIR.exists():
    moe_sys = pd.read_csv(MOE_DIR/'moe_system.csv')
    moe_cmp = pd.read_csv(MOE_DIR/'moe_components.csv')
    for dom_key, dom_pretty in [('montgomery','Montgomery'), ('shenzhen','Shenzhen'), ('tbx11k','TBX11K')]:
        rows.append({'method': 'Ours — MoE', 'source': 'eval_moe',
                     'metric': 'TB AUROC', 'dataset': dom_pretty,
                     'value': _val(moe_sys, 'tb_head_auroc', dom_key), 'notes': 'shared TB head'})
        rows.append({'method': 'Ours — MoE', 'source': 'eval_moe',
                     'metric': 'Timika AUROC', 'dataset': dom_pretty,
                     'value': _val(moe_sys, 'timika_auroc', dom_key), 'notes': 'lesion path: 4-expert MoE'})
    for dom_key, dom_pretty in [('montgomery','Montgomery'), ('shenzhen','Shenzhen')]:
        rows.append({'method': 'Ours — MoE', 'source': 'eval_moe',
                     'metric': 'Lung Dice', 'dataset': dom_pretty,
                     'value': _val(moe_cmp, 'c4_lung_dice', dom_key), 'notes': 'fine-tuned C4 decoder'})
else:
    print('WARNING: eval_moe outputs not found — skipping MoE rows. Run eval_moe.ipynb first.')

sota_df = pd.DataFrame(rows)
sota_csv = OUT_DIR / 'sota_comparison.csv'
sota_df.to_csv(sota_csv, index=False)
print(f'\nWrote {sota_csv} ({len(sota_df)} rows)')
print('\n=== SOTA COMPARISON TABLE ===')
print(sota_df.to_string(index=False))


Wrote /kaggle/working/sota/sota_comparison.csv (7 rows)

=== SOTA COMPARISON TABLE ===
                                method        source    metric    dataset    value                                                          notes
   Liu et al. 2020 (TBX11K classifier)         paper  TB AUROC     TBX11K 0.958000                                    reported best detection AUC
Liu et al. 2020 (TBX11K bbox detector)         paper   mAP@0.5     TBX11K 0.508000 TB-X box detection (different metric — referenced for context)
            MedSAM zero-shot (Ma 2024)         paper Lung Dice   Shenzhen 0.850000                        reported approx — CXR lung segmentation
            MedSAM zero-shot (Ma 2024)         paper Lung Dice Montgomery 0.850000                        reported approx — CXR lung segmentation
     Raw TXV (Cohen 2022) TB-mimic max this notebook  TB AUROC Montgomery 0.581633                no fine-tuning; max(Consolidation,Infiltration)
     Raw TXV (Cohen 2022) TB-mimic m

In [12]:
# Pivoted view — easier to drop into a paper table
import pandas as pd

pivot = sota_df.pivot_table(
    index=['method', 'source'],
    columns=['metric', 'dataset'],
    values='value',
    aggfunc='first',
)
pivot_csv = OUT_DIR / 'sota_comparison_pivot.csv'
pivot.to_csv(pivot_csv)
print(f'Wrote {pivot_csv}')
print(pivot)

Wrote /kaggle/working/sota/sota_comparison_pivot.csv
metric                                                Lung Dice           \
dataset                                              Montgomery Shenzhen   
method                                 source                              
Liu et al. 2020 (TBX11K bbox detector) paper                NaN      NaN   
Liu et al. 2020 (TBX11K classifier)    paper                NaN      NaN   
MedSAM zero-shot (Ma 2024)             paper               0.85     0.85   
Raw TXV (Cohen 2022) TB-mimic max      this notebook        NaN      NaN   

metric                                                 TB AUROC            \
dataset                                              Montgomery  Shenzhen   
method                                 source                               
Liu et al. 2020 (TBX11K bbox detector) paper                NaN       NaN   
Liu et al. 2020 (TBX11K classifier)    paper                NaN       NaN   
MedSAM zero-shot (Ma 2024)   

In [13]:
# Final: list outputs for download
print('=' * 60)
print('SOTA OUTPUTS — for download from /kaggle/working/sota/')
print('=' * 60)
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file() and p.suffix in ('.csv', '.png', '.json'):
        print(f'  {p.relative_to(OUT_DIR)}')

SOTA OUTPUTS — for download from /kaggle/working/sota/
  _nih_index_cache.json
  raw_txv_per_image.csv
  sota_comparison.csv
  sota_comparison_pivot.csv
  test_splits.json
